In [1]:
import pandas as pd
import numpy  as np
from datetime import datetime
#################################################
# Titles
#################################################

def conv(val):
    if not val:
        return 0    
    try:
        return np.int64(val)
    except:        
        return np.int64(0)
    
T = pd.read_csv('https://datasets.imdbws.com/title.basics.tsv.gz'
                 , compression='gzip'
                 , delimiter='\t'
                 , encoding='utf-8'
                 , header=0
                # , sep= " "
                 , quotechar='"'
                #, error_bad_lines=False
                 , dtype={ "tconst": "str" ,
                           "titleType": "str" ,
                           "primaryTitle": "str" ,
                           "originalTitle": "str" ,
                           "isAdult": "str", 
                         # "startYear":   "str" , 
                         # "endYear":   "str" ,
                           "runtimeMinutes": "str" ,
                           "genres": "str"             
                         }
                 , converters={'startYear':conv,'endYear':conv}                
                ) 
T["flg_doc"]           = np.where (T['genres'].str.contains('Documentary'),'Documentary'
                                 , np.where (T['genres'].str.contains('Animation'),'Animation','Fiction'))
T["primaryTitle_year"] = T['primaryTitle']  + ' ('+  T['startYear'].map(str)  + ')'
## filter movie 
T = T.loc[T.titleType == 'movie', :]
T = T.loc[T.startYear != 0, :]
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), T.shape)
 

2026-06-30 16:48:33 (637715, 11)


In [2]:
# T --> cln_imdb__xref_titles_principals_filtered

import os

data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
output_file = os.path.join(data_path, "dwh_titles__details.csv")

T.to_csv(output_file, encoding="utf-8")

In [3]:
# T.tail(1000)
T[T['tconst'] == "tt1745960"].head(15)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,flg_doc,primaryTitle_year
4620780,tt1745960,movie,Top Gun: Maverick,Top Gun: Maverick,0,2022,0,130,"Action,Drama",Fiction,Top Gun: Maverick (2022)


In [4]:
#################################################
# Names
#################################################

N = pd.read_csv('https://datasets.imdbws.com/name.basics.tsv.gz'
                 , compression='gzip'
                 , delimiter='\t',encoding='utf-8'
                 , header=0
                # , sep= " "
                 , quotechar='"'
                 #, error_bad_lines=False
                 , dtype={ "nconst": "str" ,
                           "primaryName": "str" ,
                    #      "birthYear": "str" ,
                    #       "deathYear": "str" ,
                           "primaryProfession": "str", 
                           "knownForTitles": "str"         
                         }
               , converters={'birthYear':conv,'deathYear':conv}   
               )

#N["primaryName_year"] = N['primaryName']  + ' ('+  N['birthYear'].map(str) + ' - ' +  N['deathYear'].map(str)  + ')'

N["primaryName_year"] = N['primaryName']  + np.where(N['birthYear']==0, ' ',  ' ('+  N['birthYear'].map(str) + ' - '  +  np.where (N['deathYear'] == 0, ' ',  N['deathYear'].map(str)) + ')')
N.head(3)


,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles,primaryName_year
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164",Fred Astaire (1899 - 1987)
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0117057",Lauren Bacall (1924 - 2014)
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452",Brigitte Bardot (1934 - 2025)


In [5]:
# n --> cln_imdb__xref_titles_principals_filtered

import os

data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
output_file = os.path.join(data_path, "dwh_principals__details.csv")

N.to_csv(output_file, encoding="utf-8")

In [6]:
#################################################
# Ratings
#################################################
R = pd.read_csv('https://datasets.imdbws.com/title.ratings.tsv.gz'
                 , compression='gzip'
                 , delimiter='\t',encoding='utf-8'
                 , header=0
                # , sep= " "
                 , quotechar='"'
                 #, error_bad_lines=False
                )

R.rename(columns = {  "averageRating": "tconst_imdb_rating"
                      , "numVotes":   "tconst_imdb_num_votes"
                      , "Const": "tconst"}
                      , inplace = True)
R.tail(3)

,tconst,tconst_imdb_rating,tconst_imdb_num_votes
1689170,tt9916850,6.0,7
1689171,tt9916852,5.7,7
1689172,tt9916880,7.6,11


In [7]:
# n --> cln_imdb__xref_titles_principals_filtered

import os

data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
output_file = os.path.join(data_path, "dwh_titles__ratings.csv")

R.to_csv(output_file, encoding="utf-8")

In [ ]:
############################################ 
### import my Ratings 
############################################ 
"""
myR = pd.DataFrame()
myR = pd.read_csv ("data/ratings.csv", encoding='latin-1')
myR.rename(columns = {  "Your Rating": "tconst_your_rating"
                      , "Num Votes":   "tconst_your_num_votes"
                      , "Const": "tconst"}
                      , inplace = True)
myR.tail(3)
"""

,tconst,tconst_your_rating,Date Rated,Title,Original Title,URL,Title Type,IMDb Rating,Runtime (mins),Year,Genres,tconst_your_num_votes,Release Date,Directors
1121,tt0278504,5,2011-01-02,Insomnia,Insomnia,https://www.imdb.com/title/tt0278504,Movie,7.2,118.0,2002,"Mystery, Drama, Thriller",337559,2002-05-24,Christopher Nolan
1122,tt0119488,10,2011-01-02,L.A. Confidential,L.A. Confidential,https://www.imdb.com/title/tt0119488,Movie,8.2,138.0,1997,"Crime, Drama, Thriller, Mystery",652941,1997-09-19,Curtis Hanson
1123,tt0482571,8,2011-01-02,The Prestige,The Prestige,https://www.imdb.com/title/tt0482571,Movie,8.5,130.0,2006,"Drama, Thriller, Mystery, Sci-Fi",1561837,2006-10-20,Christopher Nolan
